# 01 — Dataset summary

Produces **Table 1** (`tab:sample`) of the paper: per-band, per-epoch counts of
nights, visits and raft-stacked PSF stamps, with date ranges.

Also reports the distribution of `n_stack` (stars averaged per raft stamp), quoted
in Section `data:stamps`.

Reads only `../../data/shapelet_bvec/shapelet_{band}_{tag}.npz` and a random sample
of `../../data/stamps_stack_51/stamps_*.npz`. Nothing is re-fitted here.

In [1]:
import importlib.util, sys
spec = importlib.util.spec_from_file_location('plot_config', 'plot_config.py')
plot_config = importlib.util.module_from_spec(spec); spec.loader.exec_module(plot_config)
from plot_config import BANDS, DATA, bvec_path, STAMPS_DIR, apply_style

import numpy as np
import pandas as pd

apply_style()

EPOCHS = {'DP2': 'dp2all', 'Post-DP2': 'postdp2'}
print('data dir:', DATA)

data dir: /sdf/data/rubin/user/ztq1996/psf-rubin/psf_zernike/data


## Per-band, per-epoch counts

In [2]:
def fmt_date(d):
    s = str(int(d))
    return f'{s[:4]}-{s[4:6]}-{s[6:]}'

rows = []
for epoch, tag in EPOCHS.items():
    for band in BANDS:
        d = np.load(bvec_path(band, tag))
        v = d['visit']
        nights = np.unique(v // 100_000)
        rows.append(dict(epoch=epoch, band=band,
                         nights=len(nights),
                         visits=len(np.unique(v)),
                         psfs=len(v),
                         first=fmt_date(nights.min()),
                         last=fmt_date(nights.max())))

tab = pd.DataFrame(rows)

# Totals per epoch (visits and nights de-duplicated across bands)
for epoch, tag in EPOCHS.items():
    vs, ns = set(), set()
    for band in BANDS:
        v = np.load(bvec_path(band, tag))['visit']
        vs |= set(v.tolist()); ns |= set((v // 100_000).tolist())
    sub = tab[tab.epoch == epoch]
    print(f'{epoch:>9}: nights={len(ns):>4}  visits={len(vs):>6}  PSFs={sub.psfs.sum():>8,}')

print(f'\n    TOTAL: visits={tab.visits.sum():,}  PSFs={tab.psfs.sum():,}')
tab

      DP2: nights= 130  visits= 29502  PSFs= 612,511


 Post-DP2: nights=  85  visits= 41513  PSFs= 855,711

    TOTAL: visits=71,015  PSFs=1,468,222


,epoch,band,nights,visits,psfs,first,last
0,DP2,u,38,2056,42698,2025-04-24,2025-12-20
1,DP2,g,79,4280,88863,2025-04-24,2026-01-06
2,DP2,r,76,5063,104521,2025-04-24,2026-01-05
3,DP2,i,99,7899,164443,2025-04-24,2026-01-06
4,DP2,z,79,5867,121771,2025-05-15,2026-01-05
5,DP2,y,29,4337,90215,2025-05-19,2026-01-06
6,Post-DP2,u,17,1345,28112,2026-01-13,2026-05-13
7,Post-DP2,g,45,2794,58213,2026-01-02,2026-05-13
8,Post-DP2,r,51,5053,105190,2026-01-03,2026-05-13
9,Post-DP2,i,73,16544,341331,2026-01-01,2026-05-13


## Stars per raft stamp

Each stamp is the flux-normalised mean of up to 100 stars on one detector of one raft.
Sample the stamp files to characterise the actual stacking depth.

In [3]:
# NOTE: the stamp directory holds ~76k files on a shared filesystem; both the
# directory listing and each read cost ~0.5-1 s. Keep N_SAMPLE small -- 150
# files is ~3000 raft stamps, far more than enough for these summary statistics.
N_SAMPLE = 150

rng = np.random.default_rng(42)
files = sorted(STAMPS_DIR.glob('stamps_*.npz'))
sample = rng.choice(len(files), size=min(N_SAMPLE, len(files)), replace=False)

nstack, nraft, shp = [], [], set()
for i in sample:
    d = np.load(files[i], allow_pickle=True)
    nstack.append(d['n_stack']); nraft.append(len(d['n_stack']))
    shp.add(d['stamps'].shape[1:])
nstack = np.concatenate(nstack)

print(f'stamp files on disk       : {len(files):,}')
print(f'stamp shape               : {shp}')
print(f'files sampled             : {len(sample)}  ({len(nstack):,} raft stamps)')
print(f'rafts per visit (median)  : {np.median(nraft):.0f}  (range {min(nraft)}-{max(nraft)})')
print(f'stars per stamp   median  : {np.median(nstack):.0f}')
print(f'                  16-84%  : {np.percentile(nstack,16):.0f} - {np.percentile(nstack,84):.0f}')
print(f'                  range   : {nstack.min()} - {nstack.max()}')

stamp files on disk       : 75,929
stamp shape               : {(51, 51)}
files sampled             : 150  (3,075 raft stamps)
rafts per visit (median)  : 21  (range 2-21)
stars per stamp   median  : 60
                  16-84%  : 28 - 100
                  range   : 1 - 100


## LaTeX table for the paper

In [4]:
lines = []
for epoch in EPOCHS:
    sub = tab[tab.epoch == epoch]
    lines.append(r'\multicolumn{6}{l}{\textit{' + epoch + r'}} \\')
    for _, r in sub.iterrows():
        lines.append(f"\\quad ${r['band']}$ & {r['nights']} & {r['visits']:,} "
                     f"& {r['psfs']:,} & {r['first']} & {r['last']} " + r'\\')
    lines.append(f"\\quad all & --- & {sub['visits'].sum():,} & {sub['psfs'].sum():,} "
                 f"& & " + r'\\')
    lines.append(r'\addlinespace')
print('\n'.join(lines))

\multicolumn{6}{l}{\textit{DP2}} \\
\quad $u$ & 38 & 2,056 & 42,698 & 2025-04-24 & 2025-12-20 \\
\quad $g$ & 79 & 4,280 & 88,863 & 2025-04-24 & 2026-01-06 \\
\quad $r$ & 76 & 5,063 & 104,521 & 2025-04-24 & 2026-01-05 \\
\quad $i$ & 99 & 7,899 & 164,443 & 2025-04-24 & 2026-01-06 \\
\quad $z$ & 79 & 5,867 & 121,771 & 2025-05-15 & 2026-01-05 \\
\quad $y$ & 29 & 4,337 & 90,215 & 2025-05-19 & 2026-01-06 \\
\quad all & --- & 29,502 & 612,511 & & \\
\addlinespace
\multicolumn{6}{l}{\textit{Post-DP2}} \\
\quad $u$ & 17 & 1,345 & 28,112 & 2026-01-13 & 2026-05-13 \\
\quad $g$ & 45 & 2,794 & 58,213 & 2026-01-02 & 2026-05-13 \\
\quad $r$ & 51 & 5,053 & 105,190 & 2026-01-03 & 2026-05-13 \\
\quad $i$ & 73 & 16,544 & 341,331 & 2026-01-01 & 2026-05-13 \\
\quad $z$ & 58 & 9,237 & 187,323 & 2026-01-02 & 2026-05-13 \\
\quad $y$ & 35 & 6,540 & 135,542 & 2026-01-02 & 2026-04-28 \\
\quad all & --- & 41,513 & 855,711 & & \\
\addlinespace
